# Bengali Distill v2 — Student Model Runner

**Model:** Qwen/Qwen2.5-0.5B-Instruct fine-tuned on Bengali ChatML pairs via Knowledge Distillation  
**Teachers used:** Qwen (large) and Gemma-2-9B-IT  
**Method:** QLoRA (4-bit NF4, LoRA rank=16, alpha=32)  
**Training examples:** 5,387 Bengali instruction-response pairs (v2 format)

> Make sure Runtime > Change runtime type > T4 GPU is selected before running.

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install -q transformers==4.51.3 peft==0.20.0 accelerate bitsandbytes sentencepiece

## Step 2: Upload LoRA Adapter Files

Upload all files from your `final-lora-adapter-v2` folder:
- adapter_config.json
- adapter_model.safetensors
- tokenizer.json
- tokenizer_config.json
- chat_template.jinja
- README.md (optional)
- TRAINING_INFO.txt (optional)

In [ ]:
import os
import shutil

# --- Kaggle vs Colab Detection ---
KAGGLE_INPUT = "/kaggle/input"
ADAPTER_DIR = "/content/bangla-lora-adapter"

if os.path.exists(KAGGLE_INPUT):
    print("Running on Kaggle. Please upload the 'final-lora-adapter-v2' folder or zip")
    print("using the 'Upload' button in the 'Input' sidebar on the right.")
    # Look for adapter in kaggle input directories
    found = False
    for root, dirs, files in os.walk(KAGGLE_INPUT):
        if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
            ADAPTER_DIR = root
            found = True
            print(f"\n✅ Found adapter at: {ADAPTER_DIR}")
            break
    if not found:
        print("\n❌ Adapter not found in /kaggle/input/. Did you upload it?")
else:
    try:
        from google.colab import files
        os.makedirs(ADAPTER_DIR, exist_ok=True)
        print("Running on Colab. Please upload all files from the final-lora-adapter folder:")
        uploaded = files.upload()
        for fname, data in uploaded.items():
            dest = os.path.join(ADAPTER_DIR, fname)
            with open(dest, 'wb') as f:
                f.write(data)
            print(f"  Saved: {fname} ({len(data)/1024:.1f} KB)")
    except ImportError:
        print("Running locally. Make sure ADAPTER_DIR is set correctly below.")

print("\nAdapter directory contents:")
if os.path.exists(ADAPTER_DIR):
    for f in sorted(os.listdir(ADAPTER_DIR)):
        if os.path.isfile(os.path.join(ADAPTER_DIR, f)):
            size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
            print(f"  {f:45s} {size/1024:.1f} KB")
else:
    print("  (Directory not found)")


## Step 3: Load Base Model + LoRA Adapter

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

device_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow!)"
print(f"GPU: {device_info}")
print(f"Loading base model: {BASE_MODEL}")

# 4-bit NF4 quantization — matches training setup exactly
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Load tokenizer from adapter dir (may have Bengali-specific config)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print("Tokenizer loaded")

# Load quantized base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("Base model loaded (4-bit quantized)")

# Apply LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"LoRA adapter applied — Model ready! ({total_params:.1f}M parameters)")

## Step 4: Generation Helper

In [ ]:
SYSTEM_PROMPT = "আপনি একজন বাংলা ভাষায় দক্ষ সহকারী। ব্যবহারকারীর প্রশ্নের উত্তর সঠিক তথ্যসহ বিস্তারিতভাবে বাংলায় দিন।"

def generate_bangla(prompt, max_new_tokens=300, temperature=0.3, top_p=0.8, repetition_penalty=1.2):
    """Generate Bengali text from a prompt."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    try:
        input_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)
    except Exception:
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("generate_bangla() helper ready!")


## Step 5: Try It Out!

The model was trained on Bengali informational articles. Bengali prompts work best.

In [ ]:
# Edit this prompt — Bengali works best!
prompt = "বাংলাদেশের ইতিহাস সম্পর্কে বলুন।"  # Tell me about the history of Bangladesh.

print(f"Prompt: {prompt}")
print("-" * 70)
response = generate_bangla(prompt, max_new_tokens=300)
print("Response:")
print(response)

In [ ]:
prompt2 = "কৃত্রিম বুদ্ধিমত্তা কী?"  # What is artificial intelligence?

print(f"Prompt: {prompt2}")
print("-" * 70)
response2 = generate_bangla(prompt2, max_new_tokens=250, temperature=0.3)
print("Response:")
print(response2)

## Step 6: Interactive Chat Loop

In [ ]:
print("Bangla Student Model — Interactive Mode")
print("Type your prompt in Bengali (or English). Type 'quit' to stop.")
print("=" * 70)

while True:
    user_input = input("\nYour prompt: ").strip()
    if user_input.lower() in ("quit", "exit", "q"):
        print("Done!")
        break
    if not user_input:
        continue
    print("Generating...")
    resp = generate_bangla(user_input, max_new_tokens=300)
    print("\nResponse:")
    print(resp)
    print("-" * 70)

## Step 7 (Optional): Base Model vs Fine-tuned Comparison

See how much Bengali capability the LoRA adapter adds.

In [ ]:
prompt_compare = "বাংলাদেশের ইতিহাস সম্পর্কে বলুন।"

def generate_base_only(prompt, max_new_tokens=200):
    """Generate with base model only (LoRA weights are bypassed via disable_adapter)."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad(), model.disable_adapter():
        out = base_model.generate(
            input_ids, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)

print("BASE MODEL (no LoRA):")
print(generate_base_only(prompt_compare))
print("\n" + "=" * 70)
print("FINE-TUNED MODEL (with Bengali LoRA):")
print(generate_bangla(prompt_compare))